In [1]:
import pandas as pd
import openai
import json
import backoff
from tqdm import tqdm

## Load data

In [6]:
# CholecT45 intrument, verb, target, and triplet dictionaries
with open('../data/CholecT45-related/dicts.json', 'r') as f:
    cholect45_dicts = json.load(f)

# Load urology annotations
cmb_all = pd.read_csv('../data/urology-related/annotations/cmb_all - cmb_pure_actions1.csv')
cmb_inst_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - instrument coverage.csv')
cmb_verb_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - action_coverage.csv')
cmb_trgt_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - anatomic_coverage.csv')

# Format columns
cmb_inst_cvrg = cmb_inst_cvrg.rename(columns={'Instruments_fix': 'instruments', 'count_fix': 'count'})
cmb_verb_cvrg = cmb_verb_cvrg.rename(columns={'action_fix': 'verb', 'count_fix': 'count'})
cmb_trgt_cvrg = cmb_trgt_cvrg.rename(columns={'tissue_fix': 'target', 'count_fix': 'count'})

## Map Urology IVT to CholecT45 IVT

In [16]:
# Get OpenAI API key
with open('../keys.json', 'r') as f:
    keys = json.load(f)
openai_api_key = keys['openai_api_key-personal']

In [ ]:
#@title Function for Initializing OpenAI and Prompting
def initOpenAI(key):
    client = openai.OpenAI(
    # This is the default and can be omitted
    api_key=key,
    )
    return client

# Function to List Available Models
def listModels(client):
    models = client.models.list()
    model_names = [model.id for model in models.data]
    return model_names

In [33]:
list(cholect45_dicts['target.txt'].values())

['gallbladder',
 'cystic_plate',
 'cystic_duct',
 'cystic_artery',
 'cystic_pedicle',
 'blood_vessel',
 'fluid',
 'abdominal_wall_cavity',
 'liver',
 'adhesion',
 'omentum',
 'peritoneum',
 'gut',
 'specimen_bag',
 'null_target']

In [56]:
def get_prompt(lst, task):
    instruments = list(cholect45_dicts['instrument.txt'].values())
    verbs = list(cholect45_dicts['verb.txt'].values())
    targets = list(cholect45_dicts['target.txt'].values())
    
    if task == 'instruments':
        # Instruments
        instruction = f"You are working in the context of verbal feedback delivered by a trainer to a trainee in a live surgery. You will be a given a list containing instruments.\
        Your task is to map this list to one of the following: {instruments}. If no valid mapping can be made, then return NONE. Simply return a single instrument that the inputted list got mapped to."

    elif task == 'verbs':
        # Verbs
        instruction = f"You are working in the context of verbal feedback delivered by a trainer to a trainee in a live surgery. You will be a given a list containing verbs.\
        Your task is to map this list to one of the following: {verbs}. If no valid mapping can be made, then return NONE. Simply return a single verb that the inputted list got mapped to."

    elif task == 'targets':
        # Targets
        instruction = f"You are working in the context of verbal feedback delivered by a trainer to a trainee in a live surgery. You will be a given a list containing targets.\
        Your task is to map this list to one of the following: {targets}. If no valid mapping can be made, then return NONE. Simply return a single target that the inputted list got mapped to."
    else:
        raise ValueError("Task must be one of 'instruments', 'verbs', or 'targets'")

    label_prompt = f"Process this list: \"{lst}\""

    return instruction, label_prompt

def labelGPT(client, model_name, instruction, label_prompt, temperature=0.0):
    @backoff.on_exception(backoff.expo, (openai.RateLimitError,
                                        openai.APIError,
                                        ConnectionResetError,
                                        json.decoder.JSONDecodeError))

    def completions_with_backoff(**kwargs):
        return client.chat.completions.create(**kwargs)

    # Prompt OpenAI
    # https://github.com/openai/openai-python
    response = completions_with_backoff(model=model_name,
                                        temperature=temperature,
                                        messages=[{"role": "system", "content": instruction},
                                                {"role":"user", "content": label_prompt}])

    gen = response.choices[0].message.content
    return gen

In [35]:
openai_client = initOpenAI(openai_api_key)
available_models = listModels(openai_client)
model_name = 'gpt-4o-mini'

In [57]:
# Instruments
instrument_mappings = {}
for i in tqdm(range(len(cmb_inst_cvrg)), desc='Instruments'):
    lst = cmb_inst_cvrg.iloc[i]['instruments']
    instruction, label_prompt = get_prompt(lst, 'instruments')
    gen = labelGPT(openai_client, model_name, instruction, label_prompt)
    instrument_mappings[lst] = gen

Instruments: 100%|██████████| 113/113 [00:53<00:00,  2.11it/s]


In [ ]:
print("Valid instrument mappings:")
print({k: v for k, v in instrument_mappings.items() if v != 'NONE'})

Valid instrument mappings:
{'[electrocautery]': 'bipolar', '[clip]': 'clipper', '[scissors]': 'scissors', '[graspers]': 'grasper', '[bipolar]': 'bipolar', '[hook]': 'hook', '[scissor]': 'scissors', '[energy device]': 'bipolar', '[electrocautery, energy devices]': 'bipolar', '[jaws]': 'grasper', '[bovie]': 'bipolar', '[scissors, left hand]': 'scissors', '[scissors, graspers]': 'scissors', '[coagulation tool]': 'bipolar', '[clips]': 'clipper', '[energy devices]': 'bipolar', '[bipolar, weck]': 'bipolar', '[monopolar]': 'bipolar', '[bipolar, monopolar]': 'bipolar', '[scissors, needle driver]': 'scissors', '[scissors, 4th arm]': 'scissors', '[needle, clip]': 'clipper', '[tenaculum]': 'grasper', '[foley, weck clip]': 'clipper', '[graspers, left hand]': 'grasper', '[bovy]': 'bipolar', '[grasper]': 'grasper', '[scissors, fourth arm]': 'scissors', '[prograsp]': 'grasper', '[graspers, electrocautery]': 'grasper'}


In [60]:
# Verbs
verb_mappings = {}
for i in tqdm(range(len(cmb_verb_cvrg)), desc='Verbs'):
    lst = cmb_verb_cvrg.iloc[i]['verb']
    instruction, label_prompt = get_prompt(lst, 'verbs')
    gen = labelGPT(openai_client, model_name, instruction, label_prompt)
    verb_mappings[lst] = gen

Verbs: 100%|██████████| 1106/1106 [08:45<00:00,  2.11it/s]


In [61]:
print("Valid verb mappings:")
print({k: v for k, v in verb_mappings.items() if v != 'NONE'})

Valid verb mappings:
{'[grab]': 'grasp', '[cut]': 'cut', '[clip]': 'clip', '[retract]': 'retract', '[Clip]': 'clip', '[grab closer]': 'grasp', '[regrab]': 'null_verb', "[don't cut]": 'cut', '[cut down]': 'cut', '[get hemostasis]': 'coagulate', '[coag]': 'coagulate', '[pull back]': 'retract', '[pull towards you]': 'retract', '[hang on, grab]': 'grasp', '[grab, grab]': 'grasp', '[pull up more]': 'retract', '[retract more]': 'retract', '[Retract more]': 'retract', '[stop, retract down]': 'retract', '[come in, retract down]': 'retract', '[pinpoint, coagulate]': 'coagulate', '[start, free up, get a better grab]': 'grasp', '[do one thing, cut]': 'cut', '[cut just above]': 'cut', "[be gentle, don't retract too much]": 'retract', '[drop, cut, incise, cut]': 'cut', '[slice]': 'cut', '[cold cut, cut more distal]': 'cut', '[cut, burn]': 'cut', '[get, cut]': 'cut', '[pull gently]': 'retract', '[sweep, make it bleed, coagulate]': 'coagulate', '[burn, coagulate]': 'coagulate', '[pull back, pull back

In [ ]:
# Targets
target_mappings = {}
for i in tqdm(range(len(cmb_trgt_cvrg)), desc='Targets'):
    lst = cmb_trgt_cvrg.iloc[i]['target']
    instruction, label_prompt = get_prompt(lst, 'targets')
    gen = labelGPT(openai_client, model_name, instruction, label_prompt)
    target_mappings[lst] = gen

In [63]:
print("Valid target mappings:")
print({k: v for k, v in target_mappings.items() if v != 'NONE'})

Valid target mappings:
{'[vein]': 'blood_vessel', '[vessel]': 'blood_vessel', '[peritoneum]': 'peritoneum', '[vessels]': 'blood_vessel', '[artery]': 'cystic_artery', '[liver]': 'liver', '[veins]': 'blood_vessel', '[bowel]': 'gut', '[vein, vein]': 'blood_vessel', 'NONE': 'null_target', '[veins, artery]': 'blood_vessel', '[vessel, vessel]': 'blood_vessel', '[arteries, veins]': 'blood_vessel', '[tissue, vein]': 'blood_vessel', '[vein, artery]': 'blood_vessel', '[pedicles]': 'cystic_pedicle', '[vas deferens, vessel]': 'blood_vessel', '[blood vessel]': 'blood_vessel', '[lateral structure, vessel]': 'blood_vessel', '[vessel, thicker vessel]': 'blood_vessel', '[prostate contour, pedicle]': 'cystic_pedicle', '[lumbar vein]': 'blood_vessel', '[tissue, vessels]': 'blood_vessel', '[NONE, body wall]': 'abdominal_wall_cavity', '[vessel loop, vein, space]': 'blood_vessel', '[iliac artery]': 'blood_vessel', '[abdominal wall]': 'abdominal_wall_cavity', '[muscle, peritoneum]': 'peritoneum', '[periprost

In [66]:
cmb_inst_cvrg['mapped_instrument'] = cmb_inst_cvrg['instruments'].map(instrument_mappings)
cmb_verb_cvrg['mapped_verb'] = cmb_verb_cvrg['verb'].map(verb_mappings)
cmb_trgt_cvrg['mapped_target'] = cmb_trgt_cvrg['target'].map(target_mappings)

In [ ]:
cmb_inst_cvrg.to_csv('../data/urology-related/annotations/cmb_inst_cvrg_mapped.csv')
cmb_verb_cvrg.to_csv('../data/urology-related/annotations/cmb_verb_cvrg_mapped.csv')
cmb_trgt_cvrg.to_csv('../data/urology-related/annotations/cmb_trgt_cvrg_mapped.csv')

## Get Valid Annotations

In [9]:
cmb_inst_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_inst_cvrg_mapped.csv')
cmb_verb_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_verb_cvrg_mapped.csv')
cmb_trgt_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_trgt_cvrg_mapped.csv')

In [9]:
instrument_mappings = cmb_inst_cvrg.set_index('instruments')['mapped_instrument'].to_dict()
verb_mappings = cmb_verb_cvrg.set_index('verb')['mapped_verb'].to_dict()
target_mappings = cmb_trgt_cvrg.set_index('target')['mapped_target'].to_dict()

In [23]:
annotations_df = cmb_all.copy()
annotations_df['cholec-instrument'] = annotations_df['instrument'].map(instrument_mappings)
annotations_df['cholec-verb'] = annotations_df['action'].map(verb_mappings)
annotations_df['cholec-target'] = annotations_df['tissue'].map(target_mappings)

annotations_df['cholec-instrument'] = annotations_df['cholec-instrument'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['cholec-verb'] = annotations_df['cholec-verb'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['cholec-target'] = annotations_df['cholec-target'].apply(lambda x: x if x != 'NONE' else None)

In [24]:
len(annotations_df.dropna(subset='cholec-instrument')), len(annotations_df.dropna(subset='cholec-verb')), len(annotations_df.dropna(subset='cholec-target'))

(159, 176, 165)

In [25]:
annotations_df.dropna(subset='cholec-instrument')['cholec-instrument'].value_counts()

cholec-instrument
bipolar     80
scissors    29
grasper     22
clipper     22
hook         6
Name: count, dtype: int64

In [26]:
annotations_df.dropna(subset='cholec-verb')['cholec-verb'].value_counts()

cholec-verb
grasp        71
cut          41
retract      32
clip         10
coagulate     9
null_verb     6
dissect       5
aspirate      1
tie           1
Name: count, dtype: int64

In [27]:
annotations_df.dropna(subset='cholec-target')['cholec-target'].value_counts()

cholec-target
blood_vessel             121
peritoneum                15
cystic_artery             10
liver                      6
gut                        5
null_target                3
abdominal_wall_cavity      3
cystic_pedicle             2
Name: count, dtype: int64

In [18]:
print(json.dumps(annotations_df.iloc[0].to_dict(), indent=4))

{
    "Dialogue": "check on the left side, just see what's uhh.. it was either...",
    "Timestamp": "9:13:17",
    "Trigger": "TRUE",
    "t_omission": "FALSE",
    "t_comission": 0.0,
    "t_warning": 1,
    "t_good_action": "FALSE",
    "t_question": "FALSE",
    "f_praise": 0,
    "f_criticism": 0,
    "f_anatomic": 1,
    "f_proecdural": 1,
    "f_technical": 0,
    "f_visual_aid": 0,
    "f_other": 0.0,
    "r_m_approve": 0.0,
    "r_m_disapprove": 0,
    "r_t_verb": 0,
    "r_t_beh": 1,
    "r_t_clarify": 0,
    "r_m_rep_identical": 0,
    "r_m_rep_similar": 0,
    "r_m_control_safety": 0,
    "r_m_control_other": 0,
    "Case": 1,
    "charLen": 62,
    "wordLen_x": 12,
    "wordLen_y": 12.0,
    "GPT4_Topic_Name": "1_Pulling, Retraction and Positioning Techniques",
    "b0_Affirmative Feedback and Inquiry": 0.0,
    "b1_Pulling, Retraction and Positioning Techniques": 1.0,
    "b2_Encouraging Continuation and Progress": 0.0,
    "b3_Guidance on Prostate and Urethra Positioning

In [ ]:
tmp_df = pd.read_csv('/home/firdavs/surgery/firdavs_work/test_results/wav2vec2/both/fb_pred_with_metadata.csv')
tmp_df

annotations_df_final = annotations_df.copy()
annotations_df_final.set_index(['Case', 'Dialogue'], inplace=True)
tmp_df.set_index(['Case', 'Dialogue'], inplace=True)
annotations_df_final['cvid'] = tmp_df['cvid']
# annotations_df_final.reset_index(inplace=True)

ValueError: cannot handle a non-unique multi-index!

In [44]:
annotations_df_final.dropna(subset='cvid')

Dialogue Trigger  \
Case Timestamp                                                              
1    10:16:34   so if you could imagine the prostate running l...   FALSE   
     10:17:29                                                okay    TRUE   
     10:30:04                                                yeah    TRUE   
     10:30:30   don't buzz blindly, see what's actually bleedi...    TRUE   
     10:30:59                        that's plenty, that's plenty    TRUE   
...                                                           ...     ...   
32   16:22:32       you are going to have to eventually take this    TRUE   
     16:15:17   this little __ in the center is going to have ...   FALSE   
     16:28:00           you can take a little bit of that fat off    TRUE   
     17:12:02         be careful dont get into the prostate there    TRUE   
     17:16:26   is that case a bottle of blue light? why do we...   FALSE   

               t_omission  t_comission  t_warning t_good_action t_question  \
Case Timestamp                                                               
1    10:16:34       FALSE          0.0          0         FALSE      FALSE   
     10:17:29       FALSE          0.0          0          TRUE      FALSE   
     10:30:04       FALSE          0.0          0         FALSE       TRUE   
     10:30:30       FALSE          1.0          0         FALSE      FALSE   
     10:30:59       FALSE          1.0          0         FALSE      FALSE   
...                   ...          ...        ...           ...        ...   
32   16:22:32       FALSE          0.0          0         FALSE       TRUE   
     16:15:17       FALSE          0.0          0         FALSE      FALSE   
     16:28:00       FALSE          0.0          0         FALSE       TRUE   
     17:12:02       FALSE          0.0          1         FALSE      FALSE   
     17:16:26       FALSE          0.0          0         FALSE      FALSE   

                f_praise  f_criticism  f_anatomic  ...  coagulation_happening  \
Case Timestamp                                     ...                          
1    10:16:34          0            0           1  ...                    0.0   
     10:17:29          0            0           0  ...                    0.0   
     10:30:04          0            0           0  ...                    0.0   
     10:30:30          0            0           0  ...                    0.0   
     10:30:59          0            0           0  ...                    0.0   
...                  ...          ...         ...  ...                    ...   
32   16:22:32          0            0           0  ...                    0.0   
     16:15:17          0            0           1  ...                    0.0   
     16:28:00          0            0           1  ...                    0.0   
     17:12:02          0            0           0  ...                    0.0   
     17:16:26          0            0           0  ...                    0.0   

                large_needle_driver_present  scissors_present  \
Case Timestamp                                                  
1    10:16:34                           0.0               0.0   
     10:17:29                           0.0               0.0   
     10:30:04                           1.0               0.0   
     10:30:30                           1.0               0.0   
     10:30:59                           1.0               0.0   
...                                     ...               ...   
32   16:22:32                           0.0               0.0   
     16:15:17                           0.0               0.0   
     16:28:00                           0.0               0.0   
     17:12:02                           1.0               0.0   
     17:16:26                           1.0               0.0   

                forceps_present  visible_bleeding_present  \
Case Timestamp                                              
1    10:16:34     

In [28]:
annotations_df.to_csv('../data/urology-related/annotations/cmb_all_mapped.csv')